In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

In [59]:
df=pd.read_csv('titanic_data_updated.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [60]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
#family size creation
df['Family_size']=df['SibSp']+df['Parch']+1
#feature and target extract
X=df.drop(['Survived'],axis=1)
y=df['Survived']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [61]:
print("Column names:", X_train.columns.tolist())
print("Type:", type(X_train))
print("Shape:", X_train.shape)


Column names: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'Family_size']
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (712, 9)


In [62]:
imputer_transfer=ColumnTransformer(
    transformers=[
        ('Age',SimpleImputer(missing_values=np.nan,strategy='mean'),['Age']),
        ('Embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('Cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)
imputer_transfer.set_output(transform='pandas')
imputer_transfer.fit(X_train)
X_train=imputer_transfer.transform(X_train)



In [63]:
X_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,Family_size
331,45.500000,S,C124,False,first,male,0,0,28.5000,1
733,23.000000,S,Missing,True,second,male,0,0,13.0000,1
382,32.000000,S,Missing,True,third,male,0,0,7.9250,1
704,26.000000,S,Missing,True,third,male,1,0,7.8542,2
813,6.000000,S,Missing,True,third,female,4,2,31.2750,7
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,0,0,7.6500,1
270,29.498846,S,Missing,True,first,male,0,0,31.0000,1
860,41.000000,S,Missing,True,third,male,2,0,14.1083,3
435,14.000000,S,B96 B98,False,first,female,1,2,120.0000,4


In [64]:
X_train.isnull().sum()

,0
Age,0
Embarked,0
Cabin,0
missingindicator_Cabin,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
Family_size,0


##Outliers handling of Fare

In [65]:
fare_Q1=X_train['Fare'].quantile(0.25)
fare_Q3=X_train['Fare'].quantile(0.75)
fare_IQR=fare_Q3-fare_Q1
fare_minimum=max(0,fare_Q1-1.5*fare_IQR)
fare_maximum=fare_Q3+1.5*fare_IQR
X_train['Fare']=X_train['Fare'].clip(fare_minimum,fare_maximum)

In [66]:
X_train['Cabin_Deck']=X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck']=X_train['Cabin'].astype(str).str[0]

In [84]:
#catagorical column tranfer
encoder_scaler=ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('embarked_sex',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex']),
        ('cabin_deck',OneHotEncoder(sparse_output=False,drop='first'),['Cabin']),
        ('age_scaler',StandardScaler(),['Age']),
        ('fare_scaler',MinMaxScaler(),['Fare'])

    ],
    remainder='passthrough',
    verbose_feature_names_out=False

)
encoder_scaler.set_output(transform='pandas')
encoder_scaler.fit(X_train)
X_train_scaled = encoder_scaler.transform(X_train)
X_test_scaled = encoder_scaler.transform(X_train)  # ✅ notun name

In [73]:
X_train_scaled

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_A14,Cabin_A16,Cabin_A19,Cabin_A23,Cabin_A24,Cabin_A26,...,Cabin_G6,Cabin_Missing,Cabin_T,Age,Fare,missingindicator_Cabin,SibSp,Parch,Family_size,Cabin_Deck
331,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.232263e+00,0.442804,False,0,0,1,C
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-5.004820e-01,0.201981,True,0,0,1,M
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.926161e-01,0.123131,True,0,0,1,M
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-2.694493e-01,0.122031,True,1,0,2,M
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-1.809667e+00,0.485920,True,4,2,7,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-6.545038e-01,0.118858,True,0,0,1,M
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,2.735977e-16,0.481647,True,0,0,1,M
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,8.857142e-01,0.219201,True,2,0,3,M
435,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-1.193580e+00,1.000000,False,1,2,4,B


In [81]:
print(X_train_scaled.columns.tolist())

['Pclass', 'Embarked_Q', 'Embarked_S', 'Sex_male', 'Cabin_A14', 'Cabin_A16', 'Cabin_A19', 'Cabin_A23', 'Cabin_A24', 'Cabin_A26', 'Cabin_A32', 'Cabin_A34', 'Cabin_A36', 'Cabin_B101', 'Cabin_B102', 'Cabin_B18', 'Cabin_B19', 'Cabin_B20', 'Cabin_B22', 'Cabin_B28', 'Cabin_B3', 'Cabin_B35', 'Cabin_B37', 'Cabin_B38', 'Cabin_B4', 'Cabin_B41', 'Cabin_B42', 'Cabin_B49', 'Cabin_B5', 'Cabin_B50', 'Cabin_B51 B53 B55', 'Cabin_B57 B59 B63 B66', 'Cabin_B58 B60', 'Cabin_B71', 'Cabin_B73', 'Cabin_B77', 'Cabin_B79', 'Cabin_B80', 'Cabin_B82 B84', 'Cabin_B94', 'Cabin_B96 B98', 'Cabin_C101', 'Cabin_C103', 'Cabin_C104', 'Cabin_C106', 'Cabin_C111', 'Cabin_C118', 'Cabin_C123', 'Cabin_C124', 'Cabin_C125', 'Cabin_C128', 'Cabin_C2', 'Cabin_C22 C26', 'Cabin_C23 C25 C27', 'Cabin_C30', 'Cabin_C32', 'Cabin_C45', 'Cabin_C46', 'Cabin_C47', 'Cabin_C49', 'Cabin_C50', 'Cabin_C52', 'Cabin_C54', 'Cabin_C62 C64', 'Cabin_C65', 'Cabin_C68', 'Cabin_C7', 'Cabin_C70', 'Cabin_C78', 'Cabin_C82', 'Cabin_C83', 'Cabin_C85', 'Cabin_C86

In [85]:
X_train_scaled.drop(['Cabin_Deck', 'SibSp', 'Parch'], axis=1, inplace=True, errors='ignore')
X_test_scaled.drop(['Cabin_Deck', 'SibSp', 'Parch'], axis=1, inplace=True, errors='ignore')

In [83]:
X_train_scaled

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_A14,Cabin_A16,Cabin_A19,Cabin_A23,Cabin_A24,Cabin_A26,...,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_Missing,Cabin_T,Age,Fare,missingindicator_Cabin,Family_size
331,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.232263e+00,0.442804,False,1
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,-5.004820e-01,0.201981,True,1
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.926161e-01,0.123131,True,1
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,-2.694493e-01,0.122031,True,2
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,-1.809667e+00,0.485920,True,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,-6.545038e-01,0.118858,True,1
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,2.735977e-16,0.481647,True,1
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,8.857142e-01,0.219201,True,3
435,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-1.193580e+00,1.000000,False,4
